# CarDD 车损实例分割 · Mask R-CNN 训练

CA6129 课程项目 · Computer Vision & Applications

本 notebook 只负责 **Mask R-CNN** 这一支，跟 YOLO 那份 notebook 完全独立——两个可以分别开两次 Colab 会话跑，互不影响，谁先跑、隔多久跑都没关系。训练结果存到 Drive 后，去 `03_Compare_results.ipynb` 拼对比表格。


## 0 · 环境准备（本 notebook 独立完整，不依赖其他 notebook 跑过什么）

跟其它 notebook 完全独立：新开一个 Colab 会话，只运行这一份文件就能跑完这个模型的训练 + 评测 + 存结果。不需要提前跑过 YOLO 那份，也不需要跑过对比表格那份。

**Drive 目录结构（跟之前一样）：**

```
MyDrive/
  CarDD_project/
    cardd_code/            <- 代码文件夹（不是 zip）
    CarDD_release/         <- 官方 CarDD COCO 数据
    manifests/              <- (自动创建) manifest 缓存，生成一次后所有 notebook 和会话都复用
    colab_results/          <- (自动创建) 训练结果、权重、对比图都存这里
```

**运行时：Runtime → Change runtime type → GPU (T4 或更高)**


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# ====== 路径配置：按你 Drive 里的实际目录改这几行 ======
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/CarDD_project'
CODE_DIRNAME = 'cardd_code'
CARDD_RAW_DIRNAME = 'CarDD_release'
# ======================================================

import sys, os, json, shutil
from pathlib import Path

DRIVE_PROJECT_DIR = Path(DRIVE_PROJECT_DIR)
CODE_DIR = DRIVE_PROJECT_DIR / CODE_DIRNAME
CARDD_DRIVE_RAW_DIR = DRIVE_PROJECT_DIR / CARDD_RAW_DIRNAME
RESULTS_DRIVE_DIR = DRIVE_PROJECT_DIR / 'colab_results'
MANIFEST_DRIVE_DIR = DRIVE_PROJECT_DIR / 'manifests'
RESULTS_DRIVE_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

assert CODE_DIR.exists(), f'找不到代码文件夹: {CODE_DIR}'
assert CARDD_DRIVE_RAW_DIR.exists(), f'找不到 CarDD 数据: {CARDD_DRIVE_RAW_DIR}'

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

print('代码文件夹:', CODE_DIR)
print('结果保存位置:', RESULTS_DRIVE_DIR)
print('manifest 缓存位置:', MANIFEST_DRIVE_DIR)


代码文件夹: /content/drive/MyDrive/CarDD_project/cardd_code
结果保存位置: /content/drive/MyDrive/CarDD_project/colab_results
manifest 缓存位置: /content/drive/MyDrive/CarDD_project/manifests


In [3]:
!pip install -q -r "{CODE_DIR}/requirements.txt"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 6.5 MB/s eta 0:00:00


In [4]:
import torch, os
print('PyTorch:', torch.__version__)
print('CUDA 可用:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    YOLO_DEVICE = '0'
    TORCH_DEVICE = 'cuda'
else:
    print('警告：没有检测到 GPU，训练会很慢。去 Runtime -> Change runtime type 开 GPU。')
    YOLO_DEVICE = 'cpu'
    TORCH_DEVICE = 'cpu'

# Colab 免费版通常只有 2 个 vCPU，dataloader workers 设太多反而互相抢核；
# 按实际核数留一个给主进程，作为下面训练调用里 num_workers/workers 的默认值
NUM_WORKERS = max(1, (os.cpu_count() or 2) - 1)
print('CPU 核数:', os.cpu_count(), '-> dataloader workers 用:', NUM_WORKERS)


PyTorch: 2.11.0+cu128
CUDA 可用: True
GPU: Tesla T4
CPU 核数: 2 -> dataloader workers 用: 1


In [5]:
# 把 CarDD 原始数据从 Drive 复制一份到 Colab 本地磁盘
# Drive 挂载盘对大量小文件读取很慢，训练前拷到本地能省下不少时间
CARDD_LOCAL_RAW_DIR = Path('/content/data/raw/CarDD_release')
if not CARDD_LOCAL_RAW_DIR.exists():
    CARDD_LOCAL_RAW_DIR.parent.mkdir(parents=True, exist_ok=True)
    print('正在从 Drive 复制数据到本地磁盘，几千张图片，可能需要几分钟...')
    !cp -r "{CARDD_DRIVE_RAW_DIR}" "{CARDD_LOCAL_RAW_DIR}"
    print('复制完成')
else:
    print('本地已有数据副本，跳过复制')


正在从 Drive 复制数据到本地磁盘，几千张图片，可能需要几分钟...
复制完成


In [6]:
# manifest 现在会先看 Drive 里有没有已经生成过的缓存，有就直接复制到本地用，
# 不用每个新 session 都重新生成一次（虽然生成本来就很快，只读 JSON 标注不碰图片，
# 但既然能完全复用就没必要每次都重跑）。第一次运行会自动把生成结果也存回 Drive，
# 之后不管是这份 notebook 还是 YOLO 那份 notebook，新开多少次 session 都直接复用。
from dataset.prepare_cardd import build_manifest

MANIFEST_PATH = Path('/content/data/manifests/cardd_manifest.json')
MANIFEST_DRIVE_PATH = MANIFEST_DRIVE_DIR / 'cardd_manifest.json'
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)

def ensure_manifest():
    if MANIFEST_PATH.exists():
        print('本地已有 manifest，跳过:', MANIFEST_PATH)
    elif MANIFEST_DRIVE_PATH.exists():
        print('Drive 上已有 manifest 缓存，直接复制到本地（跨 session 复用）:', MANIFEST_DRIVE_PATH)
        shutil.copy(MANIFEST_DRIVE_PATH, MANIFEST_PATH)
    else:
        print('Drive 和本地都没有，重新生成 manifest...')
        manifest = build_manifest(CARDD_LOCAL_RAW_DIR, seed=42)
        MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
        shutil.copy(MANIFEST_PATH, MANIFEST_DRIVE_PATH)
        print('已存回 Drive，之后的 session 都会直接复用:', MANIFEST_DRIVE_PATH)
    manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
    print('数据集划分:', manifest['summary']['images'])
    print('标注数量:', manifest['summary']['annotations'])
    print('类别:', manifest['classes'])
    return manifest

manifest_preview = ensure_manifest()


Drive 和本地都没有，重新生成 manifest...
已存回 Drive，之后的 session 都会直接复用: /content/drive/MyDrive/CarDD_project/manifests/cardd_manifest.json
数据集划分: {'train': 2816, 'validation': 810, 'test': 374}
标注数量: {'train': 6211, 'validation': 1744, 'test': 785}
类别: ['dent', 'scratch', 'crack', 'glass shatter', 'lamp broken', 'tire flat']


---
## 1 · 训练 Mask R-CNN

20 epoch，800 分辨率（COCO 标准比例），seed 42，在 test split 上评测（脚本默认是 val split，这里显式改成 test 以对齐 YOLO 的评测协议）。

**这一版脚本改了什么、为什么更快（epoch 数完全没变）：**
- `use_amp=True`：脚本原来是默认关闭混合精度的，T4 上开启之后单个 epoch 通常明显更快，几乎不影响精度。
- DataLoader 加了 `pin_memory=True` + `persistent_workers=True`（已经写进 `train_mask_rcnn.py` 里，不用你在这里传参）：省掉每个 epoch 重新拉起 worker 进程、以及 host→GPU 拷贝的开销。
- `num_workers=NUM_WORKERS`：按 Colab 实际给的 CPU 核数来，而不是脚本默认的 2。
- `eval_every=2`：训练过程中的验证（用来挑 best checkpoint）从每个 epoch 一次改成每 2 个 epoch 一次，省下一半验证时间；**不影响最终报告的指标**，因为训练完之后无论如何都会显式重新在完整 eval 集上跑一次最终评测（见 897 行附近的“加载 best checkpoint 做最终评测”逻辑）。

> `batch_size` 保持默认的 2 没有改——调大 batch 会改变训练动态（等效学习率、BN 统计量等），如果你想试，把下面 `batch_size=2` 改成 `4` 自己跑一遍对比即可，但这属于超参数变化，报告里最好注明。

`images_root` 不用传：脚本会自动从 manifest 的 `source_dir` 字段读取图片位置。

In [7]:
from training.train_mask_rcnn import train as train_mask_rcnn

MASKRCNN_OUTPUT_DIR = Path('/content/runs/mask_rcnn')

mask_rcnn_summary = train_mask_rcnn(
    manifest=MANIFEST_PATH,
    output_dir=MASKRCNN_OUTPUT_DIR,
    # images_root 省略：自动从 manifest 的 source_dir 字段读取
    epochs=20,
    image_size=800,
    batch_size=2,
    num_workers=NUM_WORKERS,
    evaluation_split='test',
    eval_every=2,          # 训练期间验证间隔放宽到 2 epoch 一次，不影响最终评测结果
    seed=42,
    pretrained=True,
    device=TORCH_DEVICE,
    use_amp=(TORCH_DEVICE == 'cuda'),  # 混合精度：GPU 上开，CPU/MPS 上自动关
)
print(json.dumps({k: v for k, v in mask_rcnn_summary.items() if k != 'train_history'}, indent=2))


Evaluating on the held-out TEST split -- use only for final course reporting.
Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth


100%|██████████| 170M/170M [00:01<00:00, 166MB/s]
/content/drive/MyDrive/CarDD_project/cardd_code/training/train_mask_rcnn.py:345: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warmup_scheduler.step()


[epoch 1] step 20/1408 loss=2.4377
[epoch 1] step 40/1408 loss=1.1905
[epoch 1] step 60/1408 loss=1.2127
[epoch 1] step 80/1408 loss=0.9763
[epoch 1] step 100/1408 loss=0.9708
[epoch 1] step 120/1408 loss=1.8655
[epoch 1] step 140/1408 loss=0.6918
[epoch 1] step 160/1408 loss=0.7128
[epoch 1] step 180/1408 loss=1.4288
[epoch 1] step 200/1408 loss=0.7106
[epoch 1] step 220/1408 loss=1.0824
[epoch 1] step 240/1408 loss=0.7132
[epoch 1] step 260/1408 loss=0.9498
[epoch 1] step 280/1408 loss=0.6079
[epoch 1] step 300/1408 loss=0.5059
[epoch 1] step 320/1408 loss=0.9689
[epoch 1] step 340/1408 loss=1.2078
[epoch 1] step 360/1408 loss=0.8176
[epoch 1] step 380/1408 loss=1.0940
[epoch 1] step 400/1408 loss=0.9060
[epoch 1] step 420/1408 loss=1.0610
[epoch 1] step 440/1408 loss=0.5758
[epoch 1] step 460/1408 loss=0.7179
[epoch 1] step 480/1408 loss=0.8380
[epoch 1] step 500/1408 loss=0.7107
[epoch 1] step 520/1408 loss=1.0179
[epoch 1] step 540/1408 loss=0.3349
[epoch 1] step 560/1408 loss=1.0

In [8]:
# mean matched-pair mask IoU：训练脚本自带的评测是 AP 类型 (precision/recall/mAP)，
# 不直接给「平均 mask IoU」，这一步单独补上这个数
from evaluation.mask_iou_maskrcnn import evaluate as evaluate_mask_rcnn_mask_iou

MASKRCNN_FINAL_CKPT = MASKRCNN_OUTPUT_DIR / 'checkpoints' / 'final.pt'

mask_rcnn_mask_iou_summary = evaluate_mask_rcnn_mask_iou(
    manifest_path=MANIFEST_PATH,
    checkpoint_path=MASKRCNN_FINAL_CKPT,
    conf_threshold=0.5,
    iou_match_threshold=0.5,
    device=TORCH_DEVICE,
)
print(json.dumps(mask_rcnn_mask_iou_summary, indent=2))


{
  "evaluation_split": "test",
  "num_test_images": 374,
  "conf_threshold": 0.5,
  "iou_match_threshold": 0.5,
  "mean_mask_iou": 0.7965,
  "matched_pairs": 542,
  "unmatched_predictions": 374,
  "per_class_mean_mask_iou": {
    "dent": 0.7672,
    "scratch": 0.7397,
    "crack": 0.6862,
    "glass shatter": 0.9285,
    "lamp broken": 0.8936,
    "tire flat": 0.9511
  },
  "error_category_counts": {
    "correct": 542,
    "missed_damage": 243
  },
  "note": "Same greedy same-class matched-pair protocol as evaluation/mask_iou.py (the YOLO version), applied to the Mask R-CNN checkpoint, so both methods are directly comparable."
}


In [9]:
# 合并成对比表格要用的一行，存到 Drive（03_Compare_results.ipynb 靠这份文件拼表格）
from evaluation.merge_results import merge

(MASKRCNN_OUTPUT_DIR / 'training_summary.json').write_text(json.dumps(mask_rcnn_summary, indent=2), encoding='utf-8')
mask_rcnn_iou_path = RESULTS_DRIVE_DIR / 'mask_rcnn_mask_iou_summary.json'
mask_rcnn_iou_path.write_text(json.dumps(mask_rcnn_mask_iou_summary, indent=2), encoding='utf-8')

mask_rcnn_row = merge(
    MASKRCNN_OUTPUT_DIR / 'training_summary.json',
    mask_rcnn_iou_path,
    method_label='Mask R-CNN',
)
(RESULTS_DRIVE_DIR / 'final_results_row_mask_rcnn.json').write_text(json.dumps(mask_rcnn_row, indent=2), encoding='utf-8')
!cp "{MASKRCNN_OUTPUT_DIR}/training_summary.json" "{RESULTS_DRIVE_DIR}/mask_rcnn_training_summary.json"
!cp "{MASKRCNN_FINAL_CKPT}" "{RESULTS_DRIVE_DIR}/mask_rcnn_final.pt"

print('Mask R-CNN 结果已存到 Drive:', RESULTS_DRIVE_DIR)
mask_rcnn_row


Mask R-CNN 结果已存到 Drive: /content/drive/MyDrive/CarDD_project/colab_results


{'method': 'Mask R-CNN',
 'mask_iou': 0.7965,
 'precision': 0.6239991237587103,
 'recall': 0.7466765616770369,
 'latency_ms': 139.8582943743374,
 'mAP50': 0.7107082908094972,
 'mAP50_95': 0.5091906967130552,
 'box_precision': 0.6338506449096665,
 'box_recall': 0.7573851293198507,
 'matched_pairs': 542,
 'unmatched_predictions': 374}

---
训练 + 评测到此结束。结果已经存进 `colab_results/`，可以直接关掉这个 Colab 会话。去 `03_Compare_results.ipynb` 看对比表格（YOLO 那边没跑完也没关系，会只显示这一行）。